<a href="https://colab.research.google.com/github/mao-debug/Auto/blob/main/instanceSegmentationTutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip -q install ultralytics opencv-python matplotlib
import cv2
from ultralytics import YOLO
import matplotlib.pyplot as plt

# ---- configurable bits ----
SOURCE_VIDEO = "/Users/YE29709/Documents/Projects/SOCOM Ignite/car_vid.mp4" # path to your input video
OUTPUT_VIDEO = "output_seg2.mp4"         # path to save the annotated result
MODEL_PATH   = "yolo11x-seg.pt"         # try "yolov8n-seg.pt" if you have YOLOv8 weights
DEVICE       = "mps"                     # e.g. "cuda:0", "mps", or "cpu" (None = auto)
# ---------------------------


In [ ]:
# Load model once
model = YOLO(MODEL_PATH)
print(model.names)

{0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plant', 59: 'bed', 60: 'dining table', 61: 'toilet', 62: 'tv', 63: 'laptop', 64: 'mouse', 65: 'remote', 66: 'keyboard', 67: 'cell phone', 68: 'microw

In [ ]:
cap = cv2.VideoCapture(SOURCE_VIDEO)
if not cap.isOpened():
    raise SystemExit(f"Could not open video: {SOURCE_VIDEO}")

# Read one frame to set up the writer
ok, frame = cap.read()
if not ok:
    cap.release()
    raise SystemExit("Video appears empty.")

# Run one warm-up inference to get plotted frame size
result = model.predict(frame, verbose=False, device=DEVICE)[0]
plotted = result.plot()  # BGR image with masks/boxes drawn

# Convert BGR -> RGB and display inline in the notebook
plotted_rgb = cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(12,8))
plt.axis('off')
plt.imshow(plotted_rgb)
plt.show()


SystemExit: Could not open video: /Users/YE29709/Documents/Projects/SOCOM Ignite/car_vid.mp4

In [ ]:
h, w = plotted.shape[:2]
fps = cap.get(cv2.CAP_PROP_FPS)
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (w, h))

# write first processed frame
out.write(plotted)

In [ ]:
# Process the rest
frame_idx = 1
while True:
    ok, frame = cap.read()
    if not ok:
        break

    # Inference on this frame
    result = model.predict(frame, verbose=False, device=DEVICE)[0]
    plotted = result.plot()

    # Ensure size matches the writer (resize if needed)
    if plotted.shape[1] != w or plotted.shape[0] != h:
        plotted = cv2.resize(plotted, (w, h), interpolation=cv2.INTER_LINEAR)

    out.write(plotted)

    # (optional) quick progress
    if frame_idx % 60 == 0:
        print(f"Processed {frame_idx} frames...")
    frame_idx += 1

cap.release()
out.release()
print(f"Done. Saved to: {OUTPUT_VIDEO}")
